### 1. Kütüphanelerin Yüklenmesi

Gerekli veri işleme, makine öğrenmesi ve görselleştirme kütüphanelerini içe aktarıyoruz.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    accuracy_score,
    roc_auc_score
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

### 2. Veri Setinin Yüklenmesi

Temizlenmiş veri setini `data/processed/dataset_clean.csv` dosyasından okuyup `df` adlı değişkene (DataFrame) aktarıyoruz.

In [ ]:
df = pd.read_csv('../data/processed/dataset_clean.csv')
df.head()

### 3. Özelliklerin (X) ve Hedef Değişkenin (y) Ayrılması

Modelin tahmin etmeye çalışacağı hedef değişkeni (`target`) ve eğitim için kullanılacak özellikleri (`X`) ayırıyoruz. `target` ve model eğitimi sırasında sızıntıya yol açabilecek `final_result` sütunlarını özelliklerden çıkarıyoruz.

In [ ]:
X = df.drop(columns=['target', 'final_result'])
y = df['target']

### 4. Boyutların Kontrolü ve Sınıf Dağılımı

Oluşturulan `X` ve `y` veri setlerinin boyutlarını ekrana yazdırıyoruz ve hedef değişkenin (Pass=0, Fail=1, Distinction=2, Withdrawn=3) sınıf dağılımını inceliyoruz.

In [ ]:
print("X'in boyutu:", X.shape)
print("y'nin boyutu:", y.shape)
print("\nHedef Değişken (y) Sınıf Dağılımı:")
print(y.value_counts())

### 5. Eğitim ve Test Setlerinin Ayrılması

Modeli eğitmek ve değerlendirmek için veriyi `%80` eğitim, `%20` test olacak şekilde ayırıyoruz. Veri dengesiz olduğu için `stratify=y` parametresini kullanarak sınıfların her iki sette de aynı oranda dağılmasını sağlıyoruz.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Eğitim seti boyutu:", X_train.shape)
print("Test seti boyutu:", X_test.shape)
print("\nEğitim Seti Sınıf Dağılımı:")
print(y_train.value_counts())
print("\nTest Seti Sınıf Dağılımı:")
print(y_test.value_counts())

### 6. SMOTE ile Veri Dengeleme

Sınıf dengesizliğini gidermek için SMOTE (Synthetic Minority Over-sampling Technique) uyguluyoruz.
**ÖNEMLİ:** SMOTE işlemini *sadece eğitim verisine (training set)* uyguluyoruz. Test verisine uygulamıyoruz çünkü modelin gerçek dünyadaki performansını sentetik olmayan, orijinal dağılımdaki verilerle ölçmemiz gerekir. Aksi takdirde veri sızıntısı (data leakage) yaşanır ve metrikler yanıltıcı olur.

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("SMOTE Öncesi Eğitim Seti Sınıf Dağılımı:")
print(y_train.value_counts())
print("\nSMOTE Sonrası Eğitim Seti Sınıf Dağılımı:")
print(y_train_res.value_counts())

### 7. SMOTE Öncesi ve Sonrası Karşılaştırması

SMOTE işleminin etkisini görselleştirmek için işlem öncesi ve sonrası sınıf dağılımlarını yan yana gösteren bir çubuk grafiği (bar chart) çizdiriyoruz ve sonucu `visuals/eda/smote_karsilastirma.png` olarak kaydediyoruz.

In [ ]:
import os
os.makedirs('../visuals/eda', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x=y_train, ax=axes[0], palette='viridis')
axes[0].set_title('SMOTE Öncesi Sınıf Dağılımı')
axes[0].set_xlabel('Sınıflar')
axes[0].set_ylabel('Gözlem Sayısı')

sns.countplot(x=y_train_res, ax=axes[1], palette='viridis')
axes[1].set_title('SMOTE Sonrası Sınıf Dağılımı')
axes[1].set_xlabel('Sınıflar')
axes[1].set_ylabel('Gözlem Sayısı')

plt.tight_layout()
plt.savefig('../visuals/eda/smote_karsilastirma.png')
plt.show()

### 8. Lojistik Regresyon Modeli Eğitimi

Çok sınıflı sınıflandırma için bir Lojistik Regresyon modeli kuruyoruz. `class_weight='balanced'` ile sınıflar arası olası küçük dengesizliklere karşı ekstra bir önlem alıyoruz ve modeli SMOTE ile dengelenmiş verilerimiz (`X_train_res`, `y_train_res`) üzerinde eğitiyoruz.

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', multi_class='multinomial')
lr_model.fit(X_train_res, y_train_res)

### 9. Test Seti Üzerinde Tahmin ve Değerlendirme

Eğitilmiş modelimizi test seti (`X_test`) üzerinde tahmin yapmak için kullanıyoruz. Daha sonra modelin performansını `classification_report`, `accuracy_score` ve `f1_score` (macro) gibi metriklerle değerlendiriyoruz.

In [ ]:
y_pred_lr = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)

target_names = ['Pass', 'Fail', 'Distinction', 'Withdrawn']

print("Lojistik Regresyon Sınıflandırma Raporu:\n")
print(classification_report(y_test, y_pred_lr, target_names=target_names))

acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average='macro')
roc_auc_lr = roc_auc_score(y_test, y_pred_proba_lr, multi_class='ovr', average='macro')

print(f"Accuracy Score: {acc_lr:.4f}")
print(f"Macro F1 Score: {f1_lr:.4f}")
print(f"ROC AUC Score: {roc_auc_lr:.4f}")

### 10. Karmaşıklık Matrisi (Confusion Matrix) Görselleştirmesi

Modelin hangi sınıfları doğru, hangilerini yanlış tahmin ettiğini daha net görmek için bir karmaşıklık matrisi (confusion matrix) çizdiriyoruz ve sonucu `visuals/eda/lr_confusion_matrix.png` olarak kaydediyoruz.

In [ ]:
import os
os.makedirs('../visuals/eda', exist_ok=True)

cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=target_names)

fig, ax = plt.subplots(figsize=(8, 6))
disp_lr.plot(cmap='Blues', ax=ax, xticks_rotation=45)
ax.set_title('Lojistik Regresyon Karmaşıklık Matrisi')
plt.tight_layout()
plt.savefig('../visuals/eda/lr_confusion_matrix.png')
plt.show()

### 11. Model Sonuçlarının Kaydedilmesi

İleride diğer modellerle karşılaştırma yapabilmek adına, elde ettiğimiz başarı metriklerini bir sözlük (`dict`) içerisine kaydediyoruz.

In [ ]:
lr_results = {
    'model': 'Logistic Regression',
    'accuracy': acc_lr,
    'f1_macro': f1_lr,
    'roc_auc': roc_auc_lr
}

print("Sonuçlar başarıyla kaydedildi:")
print(lr_results)

### 12. Random Forest Modeli Eğitimi

Ağaç tabanlı bir topluluk algoritması olan Random Forest modelini kuruyoruz ve `class_weight='balanced'` parametresi ile eğitiyoruz.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_model.fit(X_train_res, y_train_res)

### 13. Test Seti Üzerinde Tahmin ve Değerlendirme (Random Forest)

Eğitilmiş Random Forest modelimizle test seti üzerinde tahminler yapıyor ve başarı metriklerini hesaplıyoruz.

In [ ]:
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)

print("Random Forest Sınıflandırma Raporu:\n")
print(classification_report(y_test, y_pred_rf, target_names=target_names))

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='macro')
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf, multi_class='ovr', average='macro')

print(f"Accuracy Score: {acc_rf:.4f}")
print(f"Macro F1 Score: {f1_rf:.4f}")
print(f"ROC AUC Score: {roc_auc_rf:.4f}")

### 14. Karmaşıklık Matrisi Görselleştirmesi - Random Forest

Random Forest modeline ait karmaşıklık matrisini çizdirip kaydediyoruz.

In [ ]:
import os
os.makedirs('../visuals/eda', exist_ok=True)

cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=target_names)

fig, ax = plt.subplots(figsize=(8, 6))
disp_rf.plot(cmap='Blues', ax=ax, xticks_rotation=45)
ax.set_title('Random Forest Karmaşıklık Matrisi')
plt.tight_layout()
plt.savefig('../visuals/eda/rf_confusion_matrix.png')
plt.show()

### 15. Model Sonuçlarının Kaydedilmesi - Random Forest

Random Forest modelinin performans metriklerini `rf_results` isimli sözlüğe ekliyoruz.

In [ ]:
rf_results = {
    'model': 'Random Forest',
    'accuracy': acc_rf,
    'f1_macro': f1_rf,
    'roc_auc': roc_auc_rf
}

print("Sonuçlar başarıyla kaydedildi:")
print(rf_results)

### 16. XGBoost Kütüphanesi Kontrolü

Modeli kurmadan önce `xgboost` kütüphanesinin yüklü olup olmadığını kontrol ediyoruz.

In [ ]:
try:
    import xgboost as xgb
    print("XGBoost başarıyla içe aktarıldı.")
except ImportError:
    print("Lütfen terminalde veya yeni bir hücrede şu komutu çalıştırın: !pip install xgboost")

### 17. XGBoost Modeli Eğitimi

Güçlü bir ağaç tabanlı topluluk modeli olan XGBoost'u (eXtreme Gradient Boosting) tanımlayıp SMOTE uygulanmış eğitim seti üzerinde eğitiyoruz.

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1)
xgb_model.fit(X_train_res, y_train_res)

### 18. Test Seti Üzerinde Tahmin ve Değerlendirme (XGBoost)

Eğitilmiş XGBoost modelimizle test seti üzerinde tahminler yapıyor ve başarı metriklerini hesaplıyoruz.

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)

print("XGBoost Sınıflandırma Raporu:\n")
print(classification_report(y_test, y_pred_xgb, target_names=target_names))

acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb, average='macro')
roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb, multi_class='ovr', average='macro')

print(f"Accuracy Score: {acc_xgb:.4f}")
print(f"Macro F1 Score: {f1_xgb:.4f}")
print(f"ROC AUC Score: {roc_auc_xgb:.4f}")

### 19. Karmaşıklık Matrisi (Confusion Matrix) Görselleştirmesi - XGBoost

XGBoost modeline ait karmaşıklık matrisini çizdirip kaydediyoruz.

In [ ]:
import os
os.makedirs('../visuals/eda', exist_ok=True)

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
disp_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=target_names)

fig, ax = plt.subplots(figsize=(8, 6))
disp_xgb.plot(cmap='Blues', ax=ax, xticks_rotation=45)
ax.set_title('XGBoost Karmaşıklık Matrisi')
plt.tight_layout()
plt.savefig('../visuals/eda/xgb_confusion_matrix.png')
plt.show()

### 20. Model Sonuçlarının Kaydedilmesi - XGBoost

XGBoost modelinin performans metriklerini daha sonra karşılaştırmak üzere `xgb_results` isimli sözlüğe ekliyoruz.

In [ ]:
xgb_results = {
    'model': 'XGBoost',
    'accuracy': acc_xgb,
    'f1_macro': f1_xgb,
    'roc_auc': roc_auc_xgb
}

print("Sonuçlar başarıyla kaydedildi:")
print(xgb_results)

### 21. Sonuçların Karşılaştırılması

Farklı modellerden (Lojistik Regresyon, Random Forest, XGBoost) elde ettiğimiz metrikleri tek bir veri çerçevesinde (DataFrame) toplayarak genel bir tablo oluşturuyoruz.

In [ ]:
results_df = pd.DataFrame([lr_results, rf_results, xgb_results])
results_df = results_df.rename(columns={
    'model': 'Model', 
    'accuracy': 'Accuracy', 
    'f1_macro': 'F1 Macro', 
    'roc_auc': 'ROC-AUC'
})
results_df = results_df.round(4)
results_df

### 22. Model Performanslarının Görselleştirilmesi

Modellerin `Accuracy`, `F1 Macro` ve `ROC-AUC` skorlarını yan yana görebilmek için gruplandırılmış bir sütun grafiği çizdiriyoruz.

In [ ]:
df_melted = results_df.melt(id_vars='Model', var_name='Metrik', value_name='Skor')

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=df_melted, x='Metrik', y='Skor', hue='Model', palette='Set2', ax=ax)

ax.set_title('Model Performans Karşılaştırması')
ax.set_xlabel('Metrikler')
ax.set_ylabel('Skorlar')
ax.set_ylim(0, 1.05)

plt.legend(title='Modeller', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../visuals/eda/model_karsilastirma.png')
plt.show()

### 23. Sonuçların Değerlendirilmesi

- **En İyi Performans Gösteren Model:** Genellikle ağaç tabanlı ve topluluk (ensemble) algoritmaları olan XGBoost ve Random Forest, lineer olan Lojistik Regresyon'a kıyasla verideki karmaşık, doğrusal olmayan ilişkileri daha iyi yakaladığı için daha yüksek başarı gösterir.
- **En Anlamlı Metrik:** Veri dengesizliği (class imbalance) problemimiz olduğu için sadece Doğruluk (Accuracy) metriğine bakmak yanıltıcıdır. Çok az görülen bir sınıfı sürekli yanlış tahmin eden bir modelin bile Accuracy değeri yüksek çıkabilir. Bu nedenle azınlık sınıfların da performansını eşit ağırlıkta hesaba katan **F1 Macro** ve **ROC-AUC** metrikleri bu problem için en belirleyici ölçütlerdir.
- **Karmaşıklık Matrisi (Confusion Matrix) Yorumu:** Matrisler incelendiğinde, belirli sınıfların (özellikle Fail ve Withdrawn gibi) birbiriyle karıştırıldığı görülebilir. Modellerin bu ayrımı yapmakta zorlanması, ilgili öğrenci davranışlarının özellik seti bağlamında çok benzer olduğunu göstermektedir.

In [ ]:
print("Modelleme tamamlandı. Sıradaki adım: Değerlendirme ve Görselleştirme (04_evaluation.ipynb)")